# Fraud Detection — 03. Инференс и сабмишены
# Fraud Detection — 03. Inference & Submission

Берём обученные модели из ноутбука 02 (`models/*.pkl`) и тестовые данные
(`data/processed/test.parquet`), считаем вероятности мошенничества и готовим файлы для
Kaggle: три одиночные модели и их ансамбль (rank-average).

We take the trained models from notebook 02 (`models/*.pkl`) and the test data
(`data/processed/test.parquet`), compute fraud probabilities and prepare Kaggle files:
three single models and their ensemble (rank-average).

## Настройка / Setup

In [1]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
from scipy.stats import rankdata

## Загрузка моделей и теста / Load models & test

Kaggle оценивает по ROC-AUC, поэтому в сабмишен идут **вероятности** (не метки 0/1), а
порог не применяется.

Kaggle scores by ROC-AUC, so the submission contains **probabilities** (not 0/1 labels),
and no threshold is applied.

In [2]:
def find_project_root():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "models" / "lgb.pkl").exists():
            return base
    return None

PROJECT_ROOT = find_project_root()
assert PROJECT_ROOT is not None, "Не найдены models/lgb.pkl — сначала запусти ноутбук 02."
MODELS_DIR = str(PROJECT_ROOT / "models")
PROCESSED_DIR = str(PROJECT_ROOT / "data" / "processed")

test = pd.read_parquet(os.path.join(PROCESSED_DIR, "test.parquet"))

lgb_model = joblib.load(os.path.join(MODELS_DIR, "lgb.pkl"))
xgb_model = joblib.load(os.path.join(MODELS_DIR, "xgb.pkl"))
cat_model = joblib.load(os.path.join(MODELS_DIR, "cat.pkl"))

# Выравниваем колонки теста под порядок признаков модели
# Align test columns to the model's feature order
feat = list(lgb_model.feature_name_)
test = test[feat]
print(f"test: {test.shape}")

test: (506691, 471)


## Предсказания и сабмишены / Predictions & submissions

Для ансамбля усредняем **ранги** предсказаний трёх моделей (для ROC-AUC важен порядок, а
ранги устойчивы к разным шкалам вероятностей).

For the ensemble we average the **ranks** of the three models' predictions (for ROC-AUC
the order matters, and ranks are robust to different probability scales).

In [3]:
preds = {
    "lgb": lgb_model.predict_proba(test)[:, 1],
    "xgb": xgb_model.predict_proba(test)[:, 1],
    "cat": cat_model.predict_proba(test)[:, 1],
}

blend = np.mean([rankdata(preds[m]) for m in ["lgb", "xgb", "cat"]], axis=0)
blend = blend / len(blend)

SUB_DIR = os.path.join(str(PROJECT_ROOT), "submissions")
os.makedirs(SUB_DIR, exist_ok=True)

def save_submission(name, scores):
    sub = pd.DataFrame({"TransactionID": test.index, "isFraud": scores})
    path = os.path.join(SUB_DIR, name)
    sub.to_csv(path, index=False)
    print(f"saved {name}  (min {scores.min():.4f}, max {scores.max():.4f})")

for m in ["lgb", "xgb", "cat"]:
    save_submission(f"submission_{m}.csv", preds[m])
save_submission("submission_blend.csv", blend)

# Бленд только lgb+xgb (без слабого CatBoost) / lgb+xgb blend (without the weaker CatBoost)
blend_lx = np.mean([rankdata(preds[m]) for m in ["lgb", "xgb"]], axis=0)
blend_lx = blend_lx / len(blend_lx)
save_submission("submission_blend_lx.csv", blend_lx)

/home/mchunikhin/miniconda3/envs/fraud/lib/python3.10/site-packages/xgboost/core.py:751: UserWarning: [16:13:50] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


saved submission_lgb.csv  (min 0.0000, max 0.9993)
saved submission_xgb.csv  (min 0.0000, max 0.9990)
saved submission_cat.csv  (min 0.0000, max 0.9999)
saved submission_blend.csv  (min 0.0002, max 1.0000)
saved submission_blend_lx.csv  (min 0.0000, max 1.0000)


## Вывод / Summary

Готово 5 файлов в `submissions/`: три одиночные модели (`submission_lgb/xgb/cat.csv`),
объединение трёх (`submission_blend.csv`) и объединение двух сильных
(`submission_blend_lx.csv`).

Результат на лидерборде (ROC-AUC, Public / Private):

| Сабмишен | Public | Private |
|---|---|---|
| Одиночный LightGBM | 0.949379 | 0.919059 |
| LightGBM + XGBoost | 0.948949 | **0.919338** |
| Объединение с CatBoost | хуже | хуже |

Лучший результат на private дало **объединение LightGBM + XGBoost** (0.919338), но одиночный
LightGBM практически вровень (0.919059) и даже выигрывает на public, а добавление CatBoost
только ухудшает результат (CatBoost слабее). Разница между объединением и одиночной моделью
мизерная, поэтому в качестве итоговой модели мы выбираем **одну настроенную модель —
LightGBM**: она проще, быстрее и стабильнее в работе.

Скриншот со страницы Submissions на Kaggle / Screenshot of the Kaggle Submissions page:

![Kaggle Submissions leaderboard](images/kaggle_submissions.jpg)

## Summary

Five files are ready in `submissions/`: three single models (`submission_lgb/xgb/cat.csv`),
a 3-model combination (`submission_blend.csv`) and a 2-strong-model combination
(`submission_blend_lx.csv`).

Leaderboard results (ROC-AUC, Public / Private):

| Submission | Public | Private |
|---|---|---|
| Single LightGBM | 0.949379 | 0.919059 |
| LightGBM + XGBoost | 0.948949 | **0.919338** |
| Combination with CatBoost | worse | worse |

The best private result came from **combining LightGBM + XGBoost** (0.919338), but the single
LightGBM is practically on par (0.919059) and even wins on the public board, while adding
CatBoost only makes it worse (CatBoost is weaker). The difference between the combination and
the single model is tiny, so as the final model we choose **a single tuned model — LightGBM**:
it is simpler, faster and more stable in production.